In [ ]:
import numpy as np
from graspnetAPI import GraspNet
import open3d as o3d
import cv2

import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
g = GraspNet('/home/bam/graspnetAPI/graspnet', camera='realsense', split='train')

In [ ]:
g.checkDataCompleteness()

In [ ]:
sceneId = 2
camera = 'realsense'
annId = 10

In [ ]:
ids = g.getDataIds(sceneIds=[sceneId])
print("Number of views per scene: :", len(ids))
print(ids)

In [ ]:
# There are 190 Scenes (100 Train - 90 Test)
# There are 256 Different views (annotations) per scene
color = g.loadRGB(sceneId, camera, annId)
depth = g.loadDepth(sceneId, camera, annId)
mask = g.loadMask(sceneId, camera, annId)

print(color.shape)
print(depth.shape)
print(mask.shape)

In [ ]:
plt.figure(figsize=(15, 5))

# Color image
plt.subplot(1, 3, 1)
plt.imshow(color)
plt.title("Color Image")
plt.axis("off")

# Depth image — show with colormap for visualization
plt.subplot(1, 3, 2)
plt.imshow(depth, cmap='plasma')  # or 'viridis', 'inferno', etc.
plt.title("Depth Image")
plt.axis("off")

# Mask image — assume binary or label mask
plt.subplot(1, 3, 3)
plt.imshow(mask, cmap='gray')
plt.title("Mask Image")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
pose = g.loadCameraPose(sceneId, camera, 0)
print(pose)

What do I need to do in here?

- [ ] Make sure I have a clean way to access all the data in the scene
- [x] Color
- [x] Depth
- [x] Mask
- [x] Table
- [x] Object Pose
- [x] Object Mesh
- [ ] I want a way to evaluate if a grasp pose is good...

What can I do outside?

- [ ] Use open3D to create table
- [ ] Implement the Gym Env
- [ ] Tested out baseline algortihim
- [ ] Create height map function

How do I imagine using this?

Create Gym Env: the cool thing is I can use this gym for a bunch of different enviornments!

- On reset:
    - it returns a synethic depth image for the average + std of the table
    - it returns a pose for the table (I can use this to check vs the one that is calculated)
    - It returns the step() observation
- On Step:
    - given the last observation that was returned (scene, camera, annotation id) it checks if sucess
    - Sucess function should take a list of grasp poses, friction setting and then return true or false.
    - It goes to the next observation (random seeded or +1 index)
    - it returns the color, depth and mask image (just as would happen with the real robot/gazebo)
- On render:
    - it should show the poses that were sent, in an interactive open3d viewer? (you can manually set the sleep time)
    - Camera Pose should be set to be the actual camera pose.
    - Option to display the actual meshes
    - Option to display table
    - Option to show the nearest succesful grasp it was associate with?
    - I should see the finger width, etc. check for collisions, visually see if it succeeds or fails to make sure its working
    - https://www.open3d.org/docs/0.9.0/tutorial/Advanced/non_blocking_visualization.html

---

Another way is to rapidly learn, if you have a discrete action map, you can update all the values at the same time! instead of just 1-4

This will actually be great for testing! its a static baseline. easy to download, etc.

- I can also check that the algorithim is picking up the right label of object... and assert if it doesn't... then your selection code is wrong.

Testing....

Lets say it fails then what? How to prevent regression?

A function that outputs a heatmap... or evluates the goodness of a grasp (different approaches!)


HeatMap (Discretize action space)

Grasp Net 1B
    - Put all grasps into a list, loop through all the scenes (if input was stacked), and evaluate the grasps


Offline Bam Data
    - Find the action in the heatmap that is most similar to the action that was attempted (flag for exact/thresholds)
    - In case of failure, check that the value is low, in case of success, the value should be high


In Case of real Failure
    - Unit Test: Rerun on image, and make sure that the confidence for the grasp near the failure is low
    - How to learn success though? You cannot replay the exact scene...
    - Set up other similar real life scenarios (no guarentee you will find success, takes time, etc)
    - Create a synthetic scene manually using objects (perhaps a network can do this and then be fined tuned by human) and use dexnet eval
        - Cool thing is you could use same point cloud...
        - How to deal with deformed objects? Its cool beacuse you are getting the benefits of sim (analytical eval) + the benefits of real (real noisy data)
    - Manual annotate by humans about what success/bad looks like (Human annotate may be wrong though! and is costly) (you can use same image though)

    - deformed object prediction: https://repositum.tuwien.at/bitstream/20.500.12708/195069/1/Eder%20Christian%20-%202024%20-%20Pose%20Estimation%20of%20Deformable%20Objects.pdf
    - Could mabye train it in unsupervised way with all the data
    - Even just approximating scene geometry with primiative shapes...? lets assume it all becomes one solid object What if you miss grasps?
    - Could I even just get some successes though? let say I put in a primiative shsape, a table, etc. that hsould provide at least some succesful grasps...
    - Then evalaute heatmap... if there is no object near by, it may mean that it wasn't label, so skip.. you cannot say anything about it.
    - Simplest is to just try agian. and keep this failure in the dataset, and remember that any points near this should be considered a failure!
    - Humans are expensive, data is cheap. Just continue sorting trash, you will continueing running into failures, successes, it all goes into a replay buffer
    - If you change your action space alot then yes you may need to change the replay buffer... 

Sample + Score

- Sample and score is cool, beacuse you never have an issue of the graspsing representation being different...
- Ultiatemly perhaps you can try both lol, if you need to compress knowledge from one dataset to another one.

Object Detection

In Case of real Failure:
    - Classical tesla dataflywheel
    - Correctly annotate the failed case, as use that as ground truth unit test to prevent regression
    - Collect and annotate more similar cases



In [ ]:
graspnet_root =  # ROOT PATH FOR GRASPNET



# initialize a GraspNet instance  

# random change
#ss

In [ ]:
l.files

In [ ]:
l['points'].shape